In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e5/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e5/train.csv
/kaggle/input/competitions/playground-series-s6e5/test.csv


In [2]:
df=pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/train.csv")
df_test=pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/test.csv")

In [3]:
df.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [4]:
len(df.columns)

16

In [5]:
import matplotlib.pyplot as py
import seaborn as sns

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 439140 entries, 0 to 439139
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      439140 non-null  int64  
 1   Driver                  439140 non-null  object 
 2   Compound                439140 non-null  object 
 3   Race                    439140 non-null  object 
 4   Year                    439140 non-null  int64  
 5   PitStop                 439140 non-null  int64  
 6   LapNumber               439140 non-null  int64  
 7   Stint                   439140 non-null  int64  
 8   TyreLife                439140 non-null  float64
 9   Position                439140 non-null  int64  
 10  LapTime (s)             439140 non-null  float64
 11  LapTime_Delta           439140 non-null  float64
 12  Cumulative_Degradation  439140 non-null  float64
 13  RaceProgress            439140 non-null  float64
 14  Position_Change     

In [7]:
df["Year"].min(),df["Year"].max()

(2022, 2025)

In [8]:
df["Year"].astype(int)-2022  #year between 0-3 

0         0
1         3
2         0
3         1
4         0
         ..
439135    1
439136    1
439137    1
439138    1
439139    1
Name: Year, Length: 439140, dtype: int64

In [9]:
df["Compound"].unique()


array(['HARD', 'MEDIUM', 'INTERMEDIATE', 'SOFT', 'WET'], dtype=object)

In [10]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
df['Compound'] = encoder.fit_transform(df['Compound'])

In [11]:
df["Compound"].unique
# to know mapped classes
encoder.classes_
#'HARD'-0, 'INTERMEDIATE'-1, 'MEDIUM'-2, 'SOFT'-3, 'WET'-4

array(['HARD', 'INTERMEDIATE', 'MEDIUM', 'SOFT', 'WET'], dtype=object)

In [12]:
df["Race"].str.replace("Grand Prix","").unique()
encoder=LabelEncoder()

In [13]:
df["Race"]=encoder.fit_transform(df["Race"])
df["Race"].unique().max()

np.int64(25)

In [14]:
len(df["Driver"].unique())

887

In [15]:
x_train=df[[column for column in df.columns if column != 'PitNextLap' and  column != 'Driver']]
y_train=df['PitNextLap']
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(
    x_train,y_train,
    test_size=0.2,
    random_state=42
)  


In [16]:
print(x_train.info())

<class 'pandas.core.frame.DataFrame'>
Index: 351312 entries, 356074 to 121958
Data columns (total 14 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      351312 non-null  int64  
 1   Compound                351312 non-null  int64  
 2   Race                    351312 non-null  int64  
 3   Year                    351312 non-null  int64  
 4   PitStop                 351312 non-null  int64  
 5   LapNumber               351312 non-null  int64  
 6   Stint                   351312 non-null  int64  
 7   TyreLife                351312 non-null  float64
 8   Position                351312 non-null  int64  
 9   LapTime (s)             351312 non-null  float64
 10  LapTime_Delta           351312 non-null  float64
 11  Cumulative_Degradation  351312 non-null  float64
 12  RaceProgress            351312 non-null  float64
 13  Position_Change         351312 non-null  float64
dtypes: float64(6), int64

In [17]:
#Binary Classification
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
epochs=1000
learning_rate=0.01
mlp_clf=MLPClassifier(hidden_layer_sizes=[200,200,200],solver='adam',verbose=True,early_stopping=True,batch_size=1024,learning_rate='adaptive',random_state=42)
pipeline=make_pipeline(MinMaxScaler(),mlp_clf)
pipeline.fit(x_train,y_train)

accuracy=pipeline.score(x_test,y_test)

Iteration 1, loss = 0.36418987
Validation score: 0.862063
Iteration 2, loss = 0.28558723
Validation score: 0.877775
Iteration 3, loss = 0.27683204
Validation score: 0.878715
Iteration 4, loss = 0.27195030
Validation score: 0.878715
Iteration 5, loss = 0.26839413
Validation score: 0.881504
Iteration 6, loss = 0.26602724
Validation score: 0.880166
Iteration 7, loss = 0.26454115
Validation score: 0.881931
Iteration 8, loss = 0.26379808
Validation score: 0.882130
Iteration 9, loss = 0.26244187
Validation score: 0.883269
Iteration 10, loss = 0.26050284
Validation score: 0.881049
Iteration 11, loss = 0.26003078
Validation score: 0.883553
Iteration 12, loss = 0.25967228
Validation score: 0.880052
Iteration 13, loss = 0.25854281
Validation score: 0.884436
Iteration 14, loss = 0.25795818
Validation score: 0.883212
Iteration 15, loss = 0.25692066
Validation score: 0.883468
Iteration 16, loss = 0.25663764
Validation score: 0.883269
Iteration 17, loss = 0.25550338
Validation score: 0.885034
Iterat

In [18]:

x=df[[column for column in df.columns if column != 'PitNextLap' and  column != 'Driver']]
y=df['PitNextLap']
pipeline.fit(x,y)


Iteration 1, loss = 0.34884718
Validation score: 0.877032
Iteration 2, loss = 0.28238945
Validation score: 0.879697
Iteration 3, loss = 0.27435157
Validation score: 0.882725
Iteration 4, loss = 0.26978215
Validation score: 0.884775
Iteration 5, loss = 0.26661720
Validation score: 0.883796
Iteration 6, loss = 0.26360089
Validation score: 0.886847
Iteration 7, loss = 0.26178792
Validation score: 0.886437
Iteration 8, loss = 0.26085744
Validation score: 0.886027
Iteration 9, loss = 0.25897450
Validation score: 0.886437
Iteration 10, loss = 0.25824482
Validation score: 0.887280
Iteration 11, loss = 0.25699057
Validation score: 0.886073
Iteration 12, loss = 0.25636046
Validation score: 0.888487
Iteration 13, loss = 0.25552386
Validation score: 0.887325
Iteration 14, loss = 0.25493315
Validation score: 0.888145
Iteration 15, loss = 0.25399747
Validation score: 0.888487
Iteration 16, loss = 0.25301911
Validation score: 0.889261
Iteration 17, loss = 0.25232338
Validation score: 0.889693
Iterat

Pipeline(steps=[('minmaxscaler', MinMaxScaler()),
                ('mlpclassifier',
                 MLPClassifier(batch_size=1024, early_stopping=True,
                               hidden_layer_sizes=[200, 200, 200],
                               learning_rate='adaptive', random_state=42,
                               verbose=True))])

In [19]:
df_test['Year']=df_test["Year"].astype(int)-2022
df_test['Compound'] = encoder.fit_transform(df_test['Compound'])
df_test["Race"]=encoder.fit_transform(df_test["Race"])
df_test.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
0,439140,D119,2,6,1,0,21,1,21.0,4,93.387,0.280,-4.984,0.403846,0.0
1,439141,VER,2,0,1,0,24,1,24.0,1,90.867,-0.129,-1.990,0.413793,0.0
2,439142,D270,2,6,1,0,24,1,24.0,11,92.871,0.041,-8.842,0.461538,0.0
3,439143,D112,3,24,2,0,6,2,4.0,15,94.967,-19.741,8.250,0.077922,1.0
4,439144,AND,0,25,2,0,52,2,29.0,12,99.112,0.930,-20.848,0.722222,7.0


In [20]:
x_test_final=df_test[
    [
        column for column in df.columns
        if column not in ['Driver', 'PitNextLap']
    ]
]
x_test_final.head()

,id,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
0,439140,2,6,1,0,21,1,21.0,4,93.387,0.280,-4.984,0.403846,0.0
1,439141,2,0,1,0,24,1,24.0,1,90.867,-0.129,-1.990,0.413793,0.0
2,439142,2,6,1,0,24,1,24.0,11,92.871,0.041,-8.842,0.461538,0.0
3,439143,3,24,2,0,6,2,4.0,15,94.967,-19.741,8.250,0.077922,1.0
4,439144,0,25,2,0,52,2,29.0,12,99.112,0.930,-20.848,0.722222,7.0


In [21]:
df_test.shape
x_test_final.shape

(188165, 14)

In [22]:

y_pred_final = pipeline.predict(x_test_final)

submission = pd.DataFrame({
    'id': x_test_final['id'],
    'PitStop': y_pred_final.astype(int)
})

submission.to_csv('submission.csv', index=False)


In [23]:
(y_pred_final == 1).sum()

np.int64(0)

In [24]:
(df["PitNextLap"]==1).sum()
(df["PitNextLap"]==0).sum()

np.int64(351759)